# Lesson 0a: Introduction to Reinforcement Learning - Theory

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/powell-clark/reinforcement-learning/blob/main/notebooks/0a_rl_introduction_theory.ipynb)

**Author**: Powell-Clark Limited  
**Course**: Reinforcement Learning from First Principles  
**Lesson**: 0a - Introduction to RL (Theory)  
**Updated**: 2025 - Aligned with elite university standards

---

## Learning Objectives

By the end of this notebook, you will understand:
1. **What is reinforcement learning** and how it differs from supervised and unsupervised learning
2. **The agent-environment interaction loop** - the fundamental paradigm of RL
3. **Key concepts**: states, actions, rewards, policies, value functions
4. **The exploration-exploitation tradeoff** - the central challenge of RL
5. **Types of RL algorithms** - model-based vs model-free, value-based vs policy-based
6. **Real-world applications** - games, robotics, LLM alignment, recommendation systems

## Prerequisites

- **Mathematics**: Basic probability (random variables, expectation)
- **Programming**: Python basics
- **Prior ML**: Helpful but not required - we'll explain from first principles

## Table of Contents

1. [Introduction: Learning by Interaction](#1-introduction)
2. [The RL Paradigm vs Supervised Learning](#2-rl-vs-supervised-learning)
3. [The Agent-Environment Interface](#3-agent-environment-interface)
4. [Key Concepts and Terminology](#4-key-concepts)
5. [Exploration vs Exploitation](#5-exploration-exploitation)
6. [Types of RL Algorithms](#6-types-of-algorithms)
7. [Real-World Applications](#7-applications)
8. [Simple GridWorld Example](#8-gridworld-example)
9. [Summary and Next Steps](#9-summary)

## 0. Setup

First, we will install the required packages and import the necessary libraries for our implementations.

In [ ]:
# Install packages for Google Colab
import sys
if 'google.colab' in sys.modules:
    print("Running on Google Colab - installing packages...")
    !pip install gymnasium[classic-control] matplotlib seaborn numpy -q
    print("Installation complete!")

# Set random seeds for reproducibility
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print(f"Random seed set to {SEED} for reproducibility")

In [ ]:
# Import libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Tuple, List, Dict
from collections import defaultdict
import gymnasium as gym

# Configure plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print(f"NumPy version: {np.__version__}")
print(f"Matplotlib version: {plt.matplotlib.__version__}")
print(f"Gymnasium version: {gym.__version__}")
print("\n✓ All libraries imported successfully!")

## 1. Introduction: Learning by Interaction

### The Central Question

Imagine you are learning to play chess for the first time. You make moves, observe the consequences, and gradually learn which moves lead to winning and which lead to losing. This is fundamentally different from learning by memorizing a textbook of optimal moves.

**Reinforcement Learning (RL)** is learning what to do—how to map situations to actions—in order to maximize a numerical reward signal through trial and error.

### Key Characteristics of RL

1. **Learning from interaction**: The agent learns by trying actions and observing results
2. **Goal-directed learning**: There is a clear objective to maximize cumulative reward
3. **Trial and error**: The agent must discover which actions yield the most reward by trying them
4. **Delayed consequences**: Actions affect not just immediate rewards but also future situations

### Historical Context

Reinforcement learning has roots in:
- **Animal psychology** (1950s): Thorndike's Law of Effect, Skinner's operant conditioning
- **Optimal control theory** (1960s): Dynamic programming, Bellman equations
- **Temporal difference learning** (1980s): Sutton's TD-Gammon
- **Modern deep RL** (2013-present): DQN playing Atari, AlphaGo, ChatGPT alignment via RLHF

## 2. RL vs Supervised Learning vs Unsupervised Learning

Let us understand how RL differs from the other main paradigms of machine learning.

### Supervised Learning

- **Goal**: Learn a mapping from inputs to outputs from labeled examples
- **Training data**: $(x, y)$ pairs where $y$ is the correct answer
- **Example**: Image classification - given image $x$, predict label $y$ (cat/dog)
- **Feedback**: Direct supervision - "this is the correct answer"

### Unsupervised Learning  

- **Goal**: Find hidden patterns or structure in unlabeled data
- **Training data**: Just $x$ values, no labels
- **Example**: Clustering customers by purchasing behavior
- **Feedback**: No explicit feedback - discover structure

### Reinforcement Learning

- **Goal**: Learn a policy (behavior) that maximizes cumulative reward
- **Training data**: Sequences of $(s, a, r, s')$ - states, actions, rewards, next states
- **Example**: Robot learning to walk - tries movements, receives reward for forward progress
- **Feedback**: Evaluative feedback - "this action led to good/bad outcomes" (not the correct action)

### Key Distinction

The fundamental difference is the **type of feedback**:
- **Supervised**: "The correct output is $y$"
- **Reinforcement**: "That action led to reward $r$" (but maybe other actions would be better!)

This evaluative feedback means the RL agent must actively **explore** to discover which actions are best.

In [ ]:
# Visualization: Learning Paradigms Comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Supervised Learning
axes[0].text(0.5, 0.7, 'Input: Image', ha='center', fontsize=12, 
             bbox=dict(boxstyle='round', facecolor='lightblue'))
axes[0].arrow(0.5, 0.65, 0, -0.15, head_width=0.05, head_length=0.05, fc='black')
axes[0].text(0.5, 0.45, 'Model', ha='center', fontsize=12,
             bbox=dict(boxstyle='round', facecolor='lightgreen'))
axes[0].arrow(0.5, 0.4, 0, -0.15, head_width=0.05, head_length=0.05, fc='black')
axes[0].text(0.5, 0.2, 'Output: "Cat"', ha='center', fontsize=12,
             bbox=dict(boxstyle='round', facecolor='lightyellow'))
axes[0].text(0.5, 0.05, 'Label: "Cat" ✓', ha='center', fontsize=10, color='green', weight='bold')
axes[0].set_xlim(0, 1)
axes[0].set_ylim(0, 1)
axes[0].axis('off')
axes[0].set_title('Supervised Learning\n(Direct Supervision)', fontsize=13, weight='bold')

# Unsupervised Learning
axes[1].scatter([0.3, 0.35, 0.32], [0.7, 0.75, 0.72], c='blue', s=100, alpha=0.6)
axes[1].scatter([0.7, 0.68, 0.72], [0.7, 0.68, 0.73], c='red', s=100, alpha=0.6)
axes[1].text(0.3, 0.5, 'Cluster 1', ha='center', fontsize=10, color='blue', weight='bold')
axes[1].text(0.7, 0.5, 'Cluster 2', ha='center', fontsize=10, color='red', weight='bold')
axes[1].text(0.5, 0.2, 'Find patterns\nin unlabeled data', ha='center', fontsize=10)
axes[1].set_xlim(0, 1)
axes[1].set_ylim(0, 1)
axes[1].axis('off')
axes[1].set_title('Unsupervised Learning\n(Pattern Discovery)', fontsize=13, weight='bold')

# Reinforcement Learning
axes[2].text(0.5, 0.85, 'Agent', ha='center', fontsize=12,
             bbox=dict(boxstyle='round', facecolor='lightblue'))
axes[2].arrow(0.55, 0.8, 0.15, -0.2, head_width=0.04, head_length=0.04, fc='blue', ec='blue')
axes[2].text(0.75, 0.65, 'Action', ha='center', fontsize=9, color='blue')
axes[2].text(0.5, 0.5, 'Environment', ha='center', fontsize=12,
             bbox=dict(boxstyle='round', facecolor='lightgreen'))
axes[2].arrow(0.45, 0.55, -0.15, 0.2, head_width=0.04, head_length=0.04, fc='green', ec='green')
axes[2].text(0.25, 0.7, 'State\nReward', ha='center', fontsize=9, color='green')
axes[2].text(0.5, 0.2, 'Learn by trial & error', ha='center', fontsize=10)
axes[2].set_xlim(0, 1)
axes[2].set_ylim(0, 1)
axes[2].axis('off')
axes[2].set_title('Reinforcement Learning\n(Learn from Interaction)', fontsize=13, weight='bold')

plt.tight_layout()
plt.show()

print("Figure 1: Comparison of the three main machine learning paradigms")

## 3. The Agent-Environment Interface

The fundamental model of RL is the **agent-environment interaction loop**. This is the heart of all reinforcement learning.

### The Interaction Loop

At each time step $t$:

1. **Agent observes state** $S_t$ from the environment
2. **Agent selects action** $A_t$ based on its policy
3. **Environment transitions** to new state $S_{t+1}$
4. **Agent receives reward** $R_{t+1}$ indicating how good the action was
5. **Repeat** from step 1 with new state

This creates a **trajectory** or **episode**:

$$S_0, A_0, R_1, S_1, A_1, R_2, S_2, A_2, R_3, \ldots$$

### Mathematical Formulation

The agent's goal is to learn a **policy** $\pi$ that maps states to actions:

$$\pi: S \rightarrow A$$

The policy can be:
- **Deterministic**: $a = \pi(s)$ - always take the same action in a given state
- **Stochastic**: $a \sim \pi(a|s)$ - sample actions from a probability distribution

The objective is to maximize the **cumulative reward** (also called **return**):

$$G_t = R_{t+1} + R_{t+2} + R_{t+3} + \cdots = \sum_{k=0}^{\infty} R_{t+k+1}$$

Often we use a **discount factor** $\gamma \in [0,1]$ to prioritize near-term rewards:

$$G_t = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + \cdots = \sum_{k=0}^{\infty} \gamma^k R_{t+k+1}$$

### Why Discounting?

1. **Mathematical convenience**: Ensures infinite sums converge
2. **Uncertainty about future**: Future rewards are less certain
3. **Preference for immediate rewards**: Models human/animal behavior
4. **Computational stability**: Prevents unbounded value functions

In [ ]:
# Demonstration: Agent-Environment Interaction with CartPole
print("=" * 60)
print("DEMONSTRATION: Agent-Environment Interaction Loop")
print("=" * 60)
print("\nWe will create a simple environment (CartPole) and show")
print("the agent-environment interaction for one episode.\n")

# Create CartPole environment
env = gym.make('CartPole-v1')

print(f"Environment: {env.spec.id}")
print(f"Observation space: {env.observation_space}")
print(f"Action space: {env.action_space}")
print(f"\nObservation space has {env.observation_space.shape[0]} dimensions:")
print("  - Cart position")
print("  - Cart velocity")
print("  - Pole angle")
print("  - Pole angular velocity")
print(f"\nAction space has {env.action_space.n} actions: 0 (left), 1 (right)")

# Run one episode with random policy
print("\n" + "=" * 60)
print("Running one episode with RANDOM policy...")
print("=" * 60 + "\n")

state, info = env.reset(seed=SEED)
episode_reward = 0
episode_length = 0
max_steps = 100

print(f"{'Step':<6} {'Action':<8} {'Reward':<8} {'Total Reward':<15} {'Done':<6}")
print("-" * 60)

for step in range(max_steps):
    # Random policy: select action randomly
    action = env.action_space.sample()
    
    # Take action in environment
    next_state, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated
    
    # Accumulate reward
    episode_reward += reward
    episode_length += 1
    
    # Print interaction (first 10 steps and last step)
    if step < 10 or done:
        action_name = "LEFT" if action == 0 else "RIGHT"
        print(f"{step:<6} {action_name:<8} {reward:<8.1f} {episode_reward:<15.1f} {str(done):<6}")
    elif step == 10:
        print("  ... (continuing) ...")
    
    # Update state
    state = next_state
    
    if done:
        break

print("\n" + "=" * 60)
print(f"Episode finished after {episode_length} steps")
print(f"Total reward collected: {episode_reward:.1f}")
print("=" * 60)

env.close()

print("\n✓ This demonstrates the agent-environment interaction loop!")
print("  The agent takes actions, receives rewards, and observes new states.")

## 4. Key Concepts and Terminology

Let us define the fundamental concepts that we will use throughout this course.

### State ($S_t$)

The **state** is a representation of the current situation. It contains all information needed to make decisions.

- **Example (Chess)**: Board configuration, whose turn, castling rights
- **Example (Robot)**: Joint angles, velocities, object positions
- **Example (Trading)**: Stock prices, portfolio, market indicators

**Markov Property**: A state is **Markov** if it contains all relevant information:

$$P(S_{t+1} | S_t, A_t, S_{t-1}, A_{t-1}, \ldots) = P(S_{t+1} | S_t, A_t)$$

The future depends only on the present, not the past. This is a crucial assumption in RL.

### Action ($A_t$)

The **action** is what the agent does. Actions can be:
- **Discrete**: Finite set of options (e.g., {move left, move right, jump})
- **Continuous**: Real-valued vectors (e.g., robot joint torques, steering angle)

### Reward ($R_t$)

The **reward** is a scalar signal indicating how good/bad the immediate action was.

- **Example (Game)**: +1 for winning, -1 for losing, 0 otherwise
- **Example (Robot)**: +distance moved forward, -energy consumed, -damage taken

**Reward hypothesis**: All goals can be formalized as maximizing expected cumulative reward.

### Policy ($\pi$)

The **policy** is the agent's behavior—how it chooses actions given states.

- **Deterministic**: $a = \pi(s)$
- **Stochastic**: $\pi(a|s) = P(A_t = a | S_t = s)$

### Value Function ($V^\pi(s)$)

The **value function** estimates how good it is to be in a state when following policy $\pi$:

$$V^\pi(s) = \mathbb{E}_\pi \left[ \sum_{k=0}^{\infty} \gamma^k R_{t+k+1} \mid S_t = s \right]$$

"If I'm in state $s$ and follow policy $\pi$, what's my expected cumulative reward?"

### Action-Value Function ($Q^\pi(s,a)$)

The **action-value function** (or Q-function) estimates how good it is to take action $a$ in state $s$:

$$Q^\pi(s,a) = \mathbb{E}_\pi \left[ \sum_{k=0}^{\infty} \gamma^k R_{t+k+1} \mid S_t = s, A_t = a \right]$$

"If I take action $a$ in state $s$, then follow policy $\pi$, what's my expected cumulative reward?"

### Optimal Policy ($\pi^*$)

The **optimal policy** achieves the highest value in all states:

$$\pi^* = \arg\max_\pi V^\pi(s) \quad \forall s$$

The optimal value functions satisfy the **Bellman optimality equations**:

$$V^*(s) = \max_a Q^*(s,a)$$
$$Q^*(s,a) = R(s,a) + \gamma \sum_{s'} P(s'|s,a) V^*(s')$$

We will derive these in Lesson 1 on MDPs.

In [ ]:
# Visualization: Discount Factor Impact
print("Visualizing the impact of discount factor γ on future rewards...\n")

# Create reward sequence
rewards = np.ones(20)  # Constant reward of 1 at each timestep
timesteps = np.arange(len(rewards))

# Calculate discounted returns for different gamma values
gammas = [0.9, 0.95, 0.99, 1.0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Discounted reward weights
for gamma in gammas:
    weights = gamma ** timesteps
    axes[0].plot(timesteps, weights, marker='o', label=f'γ = {gamma}', linewidth=2, markersize=4)

axes[0].set_xlabel('Time Step', fontsize=12)
axes[0].set_ylabel('Discount Weight  γ^k', fontsize=12)
axes[0].set_title('How Discount Factor Weights Future Rewards', fontsize=13, weight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim([0, 1.1])

# Plot 2: Cumulative discounted return
for gamma in gammas:
    discounted_returns = np.cumsum(rewards * (gamma ** timesteps))
    axes[1].plot(timesteps, discounted_returns, marker='o', label=f'γ = {gamma}', linewidth=2, markersize=4)

axes[1].set_xlabel('Time Step', fontsize=12)
axes[1].set_ylabel('Cumulative Discounted Return', fontsize=12)
axes[1].set_title('Total Value Accumulation Over Time', fontsize=13, weight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nKey insight: Lower γ makes the agent more myopic (short-sighted).")
print("Higher γ makes the agent consider long-term consequences.")
print("γ = 1.0 gives equal weight to all future rewards (undiscounted).")

## 5. Exploration vs Exploitation

This is the **central dilemma** of reinforcement learning. The agent must balance:

### Exploitation
Use current knowledge to maximize reward:
- Choose the action that has given the best reward so far
- Safe, but might miss better options

### Exploration  
Try new actions to gain more knowledge:
- Choose actions that haven't been tried much
- Risky in the short term, but might discover better long-term strategies

### The Multi-Armed Bandit Problem

The classic formulation is the **multi-armed bandit**: Imagine $k$ slot machines ("one-armed bandits"), each with unknown payout probabilities. You have $n$ pulls. How do you maximize your total winnings?

- **Pure exploitation**: Always pull the currently-best machine → might miss a better one
- **Pure exploration**: Try all machines equally → waste pulls on bad machines  
- **Optimal strategy**: Balance exploration and exploitation

### Common Exploration Strategies

1. **ε-greedy**: 
   - With probability $\epsilon$: choose random action (explore)
   - With probability $1-\epsilon$: choose best-known action (exploit)

2. **Boltzmann (Softmax)**:
   - Choose actions probabilistically based on estimated values
   - $$P(a) = \frac{e^{Q(a)/\tau}}{\sum_{a'} e^{Q(a')/\tau}}$$
   - Temperature $\tau$ controls exploration

3. **Upper Confidence Bound (UCB)**:
   - Choose action that maximizes: $Q(a) + c\sqrt{\frac{\ln(t)}{N(a)}}$
   - Optimistic in face of uncertainty

4. **Thompson Sampling**:
   - Maintain probability distribution over action values
   - Sample from distribution to choose action

In [ ]:
# Simulation: Exploration vs Exploitation in Multi-Armed Bandit
print("=" * 70)
print("SIMULATION: Multi-Armed Bandit - Exploration vs Exploitation")
print("=" * 70)
print("\nWe have 5 slot machines with unknown payout probabilities.")
print("We'll compare different exploration strategies.\n")

class MultiArmedBandit:
    """Simple k-armed bandit environment."""
    
    def __init__(self, k: int, seed: int = 42):
        np.random.seed(seed)
        # True reward probabilities (unknown to agent)
        self.true_values = np.random.randn(k)
        self.k = k
        self.optimal_action = np.argmax(self.true_values)
        
    def pull(self, action: int) -> float:
        """Pull arm and get reward."""
        # Reward is sampled from normal distribution around true value
        reward = np.random.randn() + self.true_values[action]
        return reward

def epsilon_greedy_agent(bandit: MultiArmedBandit, n_steps: int, epsilon: float) -> Tuple[np.ndarray, np.ndarray]:
    """Run epsilon-greedy agent."""
    Q = np.zeros(bandit.k)  # Estimated action values
    N = np.zeros(bandit.k)  # Action counts
    rewards = np.zeros(n_steps)
    optimal_actions = np.zeros(n_steps)
    
    for step in range(n_steps):
        # Choose action: explore or exploit
        if np.random.rand() < epsilon:
            action = np.random.randint(bandit.k)  # Explore
        else:
            action = np.argmax(Q)  # Exploit
        
        # Take action
        reward = bandit.pull(action)
        
        # Update estimates
        N[action] += 1
        Q[action] += (reward - Q[action]) / N[action]  # Incremental average
        
        # Record
        rewards[step] = reward
        optimal_actions[step] = 1 if action == bandit.optimal_action else 0
    
    return rewards, optimal_actions

# Run experiments
k_arms = 5
n_steps = 1000
n_runs = 100  # Average over multiple runs

epsilons = [0.0, 0.01, 0.1]
all_rewards = {}
all_optimal = {}

print(f"Running {n_runs} experiments for each strategy...\n")

for eps in epsilons:
    rewards_runs = []
    optimal_runs = []
    
    for run in range(n_runs):
        bandit = MultiArmedBandit(k=k_arms, seed=SEED + run)
        rewards, optimal = epsilon_greedy_agent(bandit, n_steps, epsilon=eps)
        rewards_runs.append(rewards)
        optimal_runs.append(optimal)
    
    all_rewards[eps] = np.mean(rewards_runs, axis=0)
    all_optimal[eps] = np.mean(optimal_runs, axis=0)

# Plot results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot average rewards
for eps in epsilons:
    # Smooth with moving average
    window = 50
    smoothed = np.convolve(all_rewards[eps], np.ones(window)/window, mode='valid')
    axes[0].plot(smoothed, label=f'ε = {eps}', linewidth=2)

axes[0].set_xlabel('Steps', fontsize=12)
axes[0].set_ylabel('Average Reward', fontsize=12)
axes[0].set_title('Average Reward Over Time', fontsize=13, weight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Plot optimal action percentage
for eps in epsilons:
    # Cumulative average
    cumulative_optimal = np.cumsum(all_optimal[eps]) / (np.arange(n_steps) + 1)
    axes[1].plot(cumulative_optimal * 100, label=f'ε = {eps}', linewidth=2)

axes[1].set_xlabel('Steps', fontsize=12)
axes[1].set_ylabel('% Optimal Action', fontsize=12)
axes[1].set_title('Optimal Action Selection Rate', fontsize=13, weight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim([0, 100])

plt.tight_layout()
plt.show()

print("\nResults Analysis:")
print("-" * 70)
print(f"{'Strategy':<20} {'Avg Reward (last 100)':<25} {'Optimal % (final)'}")
print("-" * 70)
for eps in epsilons:
    avg_reward = np.mean(all_rewards[eps][-100:])
    optimal_pct = np.mean(all_optimal[eps][-100:]) * 100
    print(f"ε = {eps:<17} {avg_reward:<25.3f} {optimal_pct:.1f}%")
print("-" * 70)

print("\n✓ Key Insight: Pure exploitation (ε=0) gets stuck on suboptimal actions.")
print("  Small exploration (ε=0.01 or 0.1) discovers better actions over time!")

## 6. Types of RL Algorithms

Reinforcement learning algorithms can be categorized along several dimensions. Understanding these categories helps us choose the right algorithm for a given problem.

### Model-Based vs Model-Free

**Model-Based RL**:
- Learns a model of the environment: $P(s'|s,a)$ and $R(s,a)$
- Uses model for planning (e.g., dynamic programming, MCTS)
- **Pros**: Sample efficient, can plan ahead
- **Cons**: Model errors compound, harder to learn
- **Examples**: Dyna-Q, AlphaGo (MCTS), World Models

**Model-Free RL**:
- Learns value functions or policies directly from experience
- No explicit model of environment
- **Pros**: Simpler, works when model is hard to learn
- **Cons**: Less sample efficient
- **Examples**: Q-Learning, SARSA, Policy Gradients, PPO, DQN

### Value-Based vs Policy-Based

**Value-Based**:
- Learn value function: $V(s)$ or $Q(s,a)$
- Derive policy implicitly: $\pi(s) = \arg\max_a Q(s,a)$
- **Examples**: Q-Learning, DQN, C51

**Policy-Based**:
- Learn policy directly: $\pi_\theta(a|s)$
- Optimize policy parameters $\theta$
- **Examples**: REINFORCE, PPO, A3C

**Actor-Critic** (Hybrid):
- Learn both policy (actor) and value function (critic)
- Use critic to guide policy improvement
- **Examples**: A2C, A3C, SAC, TD3

### On-Policy vs Off-Policy

**On-Policy**:
- Learn about policy $\pi$ using data from $\pi$
- Must collect new data when policy updates
- **Examples**: SARSA, PPO, A2C

**Off-Policy**:
- Learn about policy $\pi$ using data from different policy $\mu$
- Can reuse old experience (replay buffer)
- **Examples**: Q-Learning, DQN, SAC, DDPG

### Taxonomy of Major Algorithms

```
RL Algorithms
├── Model-Based
│   ├── Dyna-Q
│   ├── MCTS (AlphaGo)
│   └── World Models (Dreamer)
│
└── Model-Free
    ├── Value-Based
    │   ├── Tabular: Q-Learning, SARSA
    │   └── Deep: DQN, Rainbow, C51
    │
    ├── Policy-Based
    │   ├── REINFORCE
    │   ├── PPO (Proximal Policy Optimization)
    │   └── TRPO (Trust Region PO)
    │
    └── Actor-Critic
        ├── A2C, A3C
        ├── SAC (Soft Actor-Critic)
        └── TD3 (Twin Delayed DDPG)
```

We will learn all of these algorithms throughout this course!

## 7. Real-World Applications

Reinforcement learning has achieved remarkable success across many domains in 2025.

### 1. Game Playing

- **AlphaGo/AlphaZero** (2016-2017): Defeated world champions at Go, chess, shogi
- **OpenAI Five** (2018): Beat professional Dota 2 teams
- **AlphaStar** (2019): Reached Grandmaster level in StarCraft II
- **Atari Games** (2013-present): DQN achieved human-level performance on many Atari games

### 2. Robotics

- **Manipulation**: Grasping, assembly, dexterous manipulation
- **Locomotion**: Quadrupeds, humanoids, bipedal walking
- **Navigation**: Autonomous drones, warehouse robots
- **Sim-to-Real**: Train in simulation, deploy to real robots

### 3. LLM Alignment and RLHF (2023-2025)

- **ChatGPT**: RLHF aligns GPT models with human preferences
- **Claude**: Constitutional AI using RLHF for safe, helpful responses
- **Gemini**: Google's LLM aligned via RLHF
- **Key technique**: Reward modeling from human preferences + PPO fine-tuning

### 4. Recommendation Systems

- **YouTube**: Video recommendations
- **Netflix**: Content recommendations
- **Spotify**: Music and podcast recommendations
- **Advantage**: Optimize long-term user engagement, not just clicks

### 5. Resource Management

- **Data Centers**: Google uses RL to reduce cooling costs by 40%
- **Power Grids**: Optimize energy distribution
- **Traffic Control**: Adaptive traffic light systems

### 6. Finance and Trading

- **Portfolio optimization**: Long-term investment strategies
- **Market making**: Automated trading
- **Risk management**: Dynamic hedging

### 7. Healthcare

- **Treatment planning**: Personalized medicine, dosage optimization
- **Clinical trials**: Adaptive trial design
- **Diagnosis**: Sequential diagnostic testing

### 8. Science and Discovery

- **Protein folding**: AlphaFold uses RL-like techniques
- **Drug discovery**: Molecular design
- **Materials science**: New material discovery
- **Nuclear fusion**: Tokamak plasma control

### Why RL? When to Use It?

Use RL when:
1. **Sequential decisions**: Actions affect future states
2. **No labeled data**: Can't use supervised learning (no "correct" actions)
3. **Trial and error**: Can simulate or safely explore
4. **Long-term optimization**: Delayed rewards, complex credit assignment
5. **Interactive environment**: Agent can take actions and observe consequences

## 8. Simple GridWorld Example

Let us implement a simple GridWorld environment to solidify our understanding of the RL framework. This will be our first hands-on RL problem!

### The GridWorld Problem

- **Grid**: 4×4 grid of cells
- **Agent**: Starts at top-left corner
- **Goal**: Reach bottom-right corner (terminal state)
- **Actions**: Up, Down, Left, Right
- **Rewards**: -1 per step (encourages reaching goal quickly), +10 at goal
- **Dynamics**: Deterministic movement (always move in chosen direction unless hitting wall)

In [ ]:
class GridWorld:
    """Simple 4x4 GridWorld environment."""
    
    def __init__(self, size: int = 4):
        self.size = size
        self.start_state = (0, 0)
        self.goal_state = (size-1, size-1)
        self.state = self.start_state
        
        # Actions: 0=Up, 1=Down, 2=Left, 3=Right
        self.actions = ['U', 'D', 'L', 'R']
        self.n_actions = len(self.actions)
    
    def reset(self) -> Tuple[int, int]:
        """Reset to start state."""
        self.state = self.start_state
        return self.state
    
    def step(self, action: int) -> Tuple[Tuple[int, int], float, bool, dict]:
        """Take action and return (next_state, reward, done, info)."""
        row, col = self.state
        
        # Move based on action (0=Up, 1=Down, 2=Left, 3=Right)
        if action == 0:  # Up
            row = max(0, row - 1)
        elif action == 1:  # Down  
            row = min(self.size - 1, row + 1)
        elif action == 2:  # Left
            col = max(0, col - 1)
        elif action == 3:  # Right
            col = min(self.size - 1, col + 1)
        
        self.state = (row, col)
        
        # Check if goal reached
        done = (self.state == self.goal_state)
        
        # Reward: -1 per step, +10 at goal
        reward = 10.0 if done else -1.0
        
        return self.state, reward, done, {}
    
    def render(self, policy=None, values=None):
        """Visualize the grid."""
        fig, ax = plt.subplots(figsize=(8, 8))
        
        # Draw grid
        for i in range(self.size + 1):
            ax.axhline(i, color='black', linewidth=2)
            ax.axvline(i, color='black', linewidth=2)
        
        # Draw states
        for row in range(self.size):
            for col in range(self.size):
                # Current position
                if (row, col) == self.state:
                    circle = plt.Circle((col + 0.5, self.size - row - 0.5), 0.3, 
                                       color='blue', alpha=0.6)
                    ax.add_patch(circle)
                    ax.text(col + 0.5, self.size - row - 0.5, 'A', 
                           ha='center', va='center', fontsize=20, weight='bold', color='white')
                
                # Goal state
                elif (row, col) == self.goal_state:
                    circle = plt.Circle((col + 0.5, self.size - row - 0.5), 0.3,
                                       color='gold', alpha=0.8)
                    ax.add_patch(circle)
                    ax.text(col + 0.5, self.size - row - 0.5, 'G',
                           ha='center', va='center', fontsize=20, weight='bold')
                
                # Show values if provided
                if values is not None:
                    value = values.get((row, col), 0)
                    ax.text(col + 0.5, self.size - row - 0.8, f'{value:.1f}',
                           ha='center', va='center', fontsize=9, color='gray')
                
                # Show policy if provided
                if policy is not None and (row, col) != self.goal_state:
                    action = policy.get((row, col), 0)
                    arrow_dict = {0: '↑', 1: '↓', 2: '←', 3: '→'}
                    ax.text(col + 0.5, self.size - row - 0.3, arrow_dict[action],
                           ha='center', va='center', fontsize=16, weight='bold', color='darkgreen')
        
        ax.set_xlim(0, self.size)
        ax.set_ylim(0, self.size)
        ax.set_aspect('equal')
        ax.axis('off')
        ax.set_title('GridWorld Environment\n(A=Agent, G=Goal)', fontsize=14, weight='bold')
        plt.tight_layout()
        plt.show()

# Create and visualize GridWorld
print("Creating 4×4 GridWorld environment...\n")
env = GridWorld(size=4)
state = env.reset()

print(f"Initial state: {state}")
print(f"Goal state: {env.goal_state}")
print(f"Available actions: {env.actions}\n")

env.render()

print("\n✓ GridWorld created! Agent (A) must reach Goal (G).")

In [ ]:
# Run random policy on GridWorld
print("=" * 60)
print("Running RANDOM POLICY on GridWorld")
print("=" * 60)
print("\nThe agent takes random actions until reaching the goal.\n")

env = GridWorld(size=4)
state = env.reset()
total_reward = 0
steps = 0
max_steps = 100

print(f"{'Step':<6} {'State':<12} {'Action':<8} {'Reward':<10} {'Total Reward':<15} {'Done'}")
print("-" * 70)

while steps < max_steps:
    # Random action
    action = np.random.randint(env.n_actions)
    action_name = env.actions[action]
    
    # Take step
    next_state, reward, done, info = env.step(action)
    total_reward += reward
    
    # Print (first 15 steps or last step)
    if steps < 15 or done:
        print(f"{steps:<6} {str(state):<12} {action_name:<8} {reward:<10.1f} {total_reward:<15.1f} {done}")
    elif steps == 15:
        print("  ... (continuing) ...")
    
    state = next_state
    steps += 1
    
    if done:
        break

print("\n" + "=" * 60)
print(f"Episode finished in {steps} steps")
print(f"Total reward: {total_reward:.1f}")
print("=" * 60)

if steps >= max_steps:
    print("\n⚠ Agent did not reach goal within max steps (random policy is inefficient!)")
else:
    print("\n✓ Agent reached goal! But random policy is very inefficient.")
    print("  We'll learn better policies in upcoming lessons!")

## 9. Summary and Next Steps

### What We Learned

In this introductory lesson, we covered the fundamental concepts of reinforcement learning:

1. **RL Paradigm**: Learning by interaction with an environment to maximize cumulative reward
2. **Key Differences**: RL uses evaluative feedback (not instructive), handles sequential decisions, and learns from trial-and-error
3. **Agent-Environment Loop**: The fundamental cycle of observation, action, and reward
4. **Core Concepts**:
   - States, actions, rewards
   - Policies (deterministic and stochastic)
   - Value functions (V and Q)
   - Discount factor γ
5. **Exploration vs Exploitation**: The central challenge - balancing trying new actions vs using current knowledge
6. **Algorithm Types**: Model-based/free, value/policy-based, on/off-policy
7. **Applications**: Games, robotics, LLM alignment (RLHF), recommendations, and more
8. **Hands-on**: Implemented and tested GridWorld and multi-armed bandit

### Key Takeaways

- **RL is learning what to do to maximize reward through trial and error**
- **The agent-environment interaction is the core framework**
- **Exploration is essential** - must try actions to discover which are best
- **Value functions predict long-term reward**, guiding action selection
- **RL is powerful for sequential decision making** where supervised learning doesn't apply

### Next Steps

Now that we understand the RL paradigm, we are ready to formalize it mathematically:

**Next Lesson**: [1a: Markov Decision Processes (MDPs)](./1a_mdp_theory.ipynb)
- Formal definition of MDPs
- Bellman equations
- Optimal policies and value functions
- Dynamic programming solutions

**Practical Next**: [0b: RL Setup and Practical Tools](./0b_rl_setup_practical.ipynb)
- Gymnasium API deep dive
- Creating custom environments
- Visualization techniques
- Debugging RL agents

### Exercises

Test your understanding:

1. **Conceptual**: Explain in your own words why RL is different from supervised learning. When would you use each?

2. **Mathematical**: Calculate the discounted return for a sequence of rewards [1, 2, 3, 4, 5] with γ = 0.9.

3. **Implementation**: Modify the GridWorld to have a "penalty" cell at (1,1) that gives -10 reward. How does this affect the optimal path?

4. **Exploration**: Implement UCB (Upper Confidence Bound) exploration for the multi-armed bandit and compare it to ε-greedy.

5. **Critical Thinking**: For what real-world problems would pure exploitation (ε=0) actually be optimal? When is exploration essential?

### Additional Resources

- **Textbook**: Sutton & Barto, "Reinforcement Learning: An Introduction" (2018), Chapter 1
- **Course**: David Silver's RL Course, Lecture 1
- **Paper**: "Reinforcement Learning: A Survey" (Kaelbling et al., 1996)
- **Blog**: OpenAI Spinning Up - "Introduction to RL"

---

**Congratulations!** You have completed Lesson 0a and understand the foundations of reinforcement learning. You are ready to dive into the mathematical formalism in Lesson 1!

### Feedback

Found an issue or have suggestions? [Open an issue on GitHub](https://github.com/powell-clark/reinforcement-learning/issues)

---

**Next**: [Lesson 1a: Markov Decision Processes →](./1a_mdp_theory.ipynb)